# Random Fixed Encoder v2 — Fixed Normalization

**Fix:** RandomFixedEncoder now normalizes inputs first (same scale as all other encoders),
then applies random projection. Without this, raw coordinates (~0-512) through random weights
produce huge values that break SIGReg.

**Also added:** `random_fixed_normalized` — random projection of the SAME 3 block coordinates
that prescribed uses, to isolate: is it the stability of the *right* subspace, or any stable subspace?

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/prescribed-axes/paper2_random_fixed_v2'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Save dir: {SAVE_DIR}')

In [ ]:
import numpy as np, json, time
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# === SIGReg ===
class SIGReg(nn.Module):
    def __init__(s, k=17, np_=512):
        super().__init__(); s.np_ = np_
        t = torch.linspace(0, 3, k); dt = 3/(k-1)
        w = torch.full((k,), 2*dt); w[[0,-1]] = dt
        phi = torch.exp(-t.square()/2.0)
        s.register_buffer('t', t); s.register_buffer('phi', phi)
        s.register_buffer('weights', w*phi)
    def forward(s, proj):
        A = torch.randn(proj.size(-1), s.np_, device=proj.device)
        A = A.div_(A.norm(p=2, dim=0))
        x = (proj @ A).unsqueeze(-1) * s.t
        err = (x.cos().mean(-3) - s.phi).square() + x.sin().mean(-3).square()
        return ((err @ s.weights) * proj.size(-2)).mean()

# === Data ===
def synth(n, seed=42):
    rng = np.random.default_rng(seed); eps = []
    for _ in range(n):
        ag = rng.uniform(50, 462, 2).astype(np.float32)
        bp = rng.uniform(100, 412, 2).astype(np.float32)
        ba = np.float32(rng.uniform(0, 2*np.pi))
        st = np.array([ag[0],ag[1],bp[0],bp[1],ba], dtype=np.float32)
        ss, aa = [st.copy()], []
        tgt = rng.uniform(50, 462, 2).astype(np.float32)
        for step in range(300):
            if step % 20 == 0: tgt = rng.uniform(50, 462, 2).astype(np.float32)
            act = np.clip(tgt + rng.normal(0, 10, 2), 0, 512).astype(np.float32)
            d = act - ag; dn = np.linalg.norm(d)
            if dn > 0: ag += d * min(1., 20./dn)
            ag = np.clip(ag, 0, 512)
            tb = bp - ag; cd = np.linalg.norm(tb)
            if 0 < cd < 30:
                f = (30-cd)/30*5; bp += (tb/cd)*f
                ba = (ba + rng.normal(0, .05)*f) % (2*np.pi)
            bp = np.clip(bp, 0, 512)
            st = np.array([ag[0],ag[1],bp[0],bp[1],ba], dtype=np.float32)
            if (step+1) % 5 == 0: ss.append(st.copy()); aa.append(act)
        if len(aa) >= 4:
            eps.append({'s': np.array(ss[:len(aa)+1]), 'a': np.array(aa)})
    return eps

class DS(Dataset):
    def __init__(s, eps, H=3):
        s.w = []
        for e in eps:
            st, a = e['s'], e['a']
            for t in range(len(a)-H):
                s.w.append((st[t:t+H+2].astype(np.float32), a[t:t+H+1].astype(np.float32)))
    def __len__(s): return len(s.w)
    def __getitem__(s, i):
        st, a = s.w[i]
        return torch.from_numpy(st), torch.from_numpy(a)

print('Infrastructure ready')

In [ ]:
# === ENCODERS (FIXED VERSION) ===

class PrescribedEncoder(nn.Module):
    """GT: extract (x_b, y_b, theta_b), normalize to ~[0,1]."""
    def __init__(s):
        super().__init__()
        s.register_buffer('sc', torch.tensor([1/512, 1/512, 1/(2*np.pi)]))
    def forward(s, x): return x[..., 2:5] * s.sc


class FreeEncoder(nn.Module):
    """Learnable MLP 5->3."""
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(
            nn.Linear(5, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 64), nn.LayerNorm(64), nn.GELU(),
            nn.Linear(64, 3))
    def forward(s, x): return s.net(x)


class RandomFixedEncoder(nn.Module):
    """FIXED: normalize ALL 5 inputs to [0,1] first, THEN random projection.
    Random orthogonal 5->3. Frozen. Stable but semantically arbitrary."""
    def __init__(s, seed=0):
        super().__init__()
        # Normalize all 5 state dimensions to ~[0,1]
        s.register_buffer('sc', torch.tensor([1/512, 1/512, 1/512, 1/512, 1/(2*np.pi)]))
        # Random orthogonal projection 5->3 via QR
        rng = torch.Generator().manual_seed(seed)
        A = torch.randn(5, 3, generator=rng)
        Q, _ = torch.linalg.qr(A)
        s.register_buffer('Q', Q)
    def forward(s, x):
        return (x * s.sc) @ s.Q


class RandomFixed3DEncoder(nn.Module):
    """Random orthogonal rotation of the SAME 3 block coordinates prescribed uses.
    This is intermediate: uses the right subspace (block coords) but random axes.
    Isolates: is it about selecting the RIGHT variables, or about any fixed projection?"""
    def __init__(s, seed=0):
        super().__init__()
        s.register_buffer('sc', torch.tensor([1/512, 1/512, 1/(2*np.pi)]))
        rng = torch.Generator().manual_seed(seed)
        A = torch.randn(3, 3, generator=rng)
        Q, _ = torch.linalg.qr(A)
        s.register_buffer('Q', Q)
    def forward(s, x):
        return (x[..., 2:5] * s.sc) @ s.Q


# Note: RandomFixed3DEncoder is identical to RotatedPrescribedEncoder.
# Keeping both names for clarity in the results table.
RotatedPrescribedEncoder = RandomFixed3DEncoder

print('Encoders ready (fixed normalization)')

In [ ]:
# === Model + Training ===

class ActionEncoder(nn.Module):
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(2, 32), nn.GELU(), nn.Linear(32, 3))
    def forward(s, a): return s.net(a)

class Predictor(nn.Module):
    def __init__(s):
        super().__init__()
        s.net = nn.Sequential(
            nn.Linear(18, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Linear(128, 128), nn.LayerNorm(128), nn.GELU(),
            nn.Linear(128, 3))
    def forward(s, e, ae):
        return s.net(torch.cat([e, ae], -1).reshape(e.size(0), -1))

class WorldModel(nn.Module):
    def __init__(s, enc, ae, pr, sig):
        super().__init__()
        s.enc, s.ae, s.pr, s.sig = enc, ae, pr, sig
    def forward(s, st, a):
        emb = s.enc(st)
        ctx, tgt = emb[:, :3], emb[:, 3]
        aem = s.ae(a[:, :3])
        p = s.pr(ctx, aem)
        return {'pl': F.mse_loss(p, tgt.detach()),
                'sl': s.sig(emb.transpose(0, 1))}

def train_one_epoch(mdl, tl, opt, lam):
    mdl.train(); tp, ts, n = 0, 0, 0
    for s, a in tl:
        o = mdl(s, a)
        l = o['pl'] + lam * o['sl']
        opt.zero_grad(); l.backward()
        nn.utils.clip_grad_norm_(mdl.parameters(), 1.0)
        opt.step()
        b = s.size(0); tp += o['pl'].item()*b; ts += o['sl'].item()*b; n += b
    return tp/n, ts/n

@torch.no_grad()
def val_loss(mdl, vl):
    mdl.eval(); tp, n = 0, 0
    for s, a in vl:
        o = mdl(s, a); tp += o['pl'].item()*s.size(0); n += s.size(0)
    return tp/n

def run_condition(encoder_type, eps, seed, epochs=30, lam=0.09):
    torch.manual_seed(seed); np.random.seed(seed)
    ds = DS(eps, 3)
    nt = int(len(ds)*0.9); nv = len(ds)-nt
    tr, va = random_split(ds, [nt, nv], generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(tr, batch_size=64, shuffle=True, drop_last=True)
    vl = DataLoader(va, batch_size=64)

    if encoder_type == 'prescribed':           enc = PrescribedEncoder()
    elif encoder_type == 'free':               enc = FreeEncoder()
    elif encoder_type == 'random_fixed':       enc = RandomFixedEncoder(seed=seed+1000)
    elif encoder_type == 'rotated_prescribed': enc = RotatedPrescribedEncoder(seed=seed+2000)
    else: raise ValueError(encoder_type)

    mdl = WorldModel(enc, ActionEncoder(), Predictor(), SIGReg())
    trainable = [p for p in mdl.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=3e-4, weight_decay=1e-3)

    hist = []
    for ep in range(1, epochs+1):
        tp, ts = train_one_epoch(mdl, tl, opt, lam)
        vp = val_loss(mdl, vl)
        hist.append({'ep': ep, 'vp': round(vp,8), 'tp': round(tp,8)})
        if ep in [1,5,10,20,30]:
            print(f'    ep={ep:>2d}  val={vp:.6f}')

    best_vp = min(h['vp'] for h in hist)
    return {'encoder': encoder_type, 'seed': seed,
            'best_val_loss': best_vp, 'final_val_loss': hist[-1]['vp'],
            'history': hist}

print('Model + training ready')

In [ ]:
# === RUN ALL ===

SEEDS = [42, 123, 777]
N_EP = 200
N_EPOCHS = 30
CONDITIONS = ['prescribed', 'rotated_prescribed', 'random_fixed', 'free']

print('='*60)
print('RANDOM FIXED ENCODER v2 — FIXED NORMALIZATION')
print('='*60)

R = {}

for seed in SEEDS:
    print(f'\n{"="*40}  SEED {seed}  {"="*40}')
    eps = synth(N_EP, seed)
    print(f'Episodes: {len(eps)}')

    for cond in CONDITIONS:
        print(f'\n  >> {cond}')
        t0 = time.time()
        res = run_condition(cond, eps, seed, N_EPOCHS)
        dt = time.time() - t0
        R[f'{cond}_seed{seed}'] = res
        print(f'  => best={res["best_val_loss"]:.6f}  final={res["final_val_loss"]:.6f}  ({dt:.0f}s)')

    # Autosave
    with open(f'{SAVE_DIR}/results_partial.json', 'w') as f:
        json.dump(R, f, indent=2)
    print(f'  [saved]')

print('\n\nDone.')

In [ ]:
# === SUMMARY ===

means = {}
print('='*70)
print('RESULTS')
print('='*70)
print(f'{"Condition":<25} {"Seed 42":>10} {"Seed 123":>10} {"Seed 777":>10} {"Mean":>10}')
print('-'*65)
for cond in CONDITIONS:
    vals = [R[f'{cond}_seed{s}']['best_val_loss'] for s in SEEDS]
    m = np.mean(vals)
    means[cond] = m
    print(f'{cond:<25} {vals[0]:>10.6f} {vals[1]:>10.6f} {vals[2]:>10.6f} {m:>10.6f}')

print(f'\n--- Ratios ---')
print(f'free / prescribed:           {means["free"]/means["prescribed"]:>8.1f}x')
print(f'random_fixed / prescribed:   {means["random_fixed"]/means["prescribed"]:>8.1f}x')
print(f'rotated / prescribed:        {means["rotated_prescribed"]/means["prescribed"]:>8.1f}x')
print(f'free / random_fixed:         {means["free"]/means["random_fixed"]:>8.1f}x')

In [ ]:
# === INTERPRETATION ===

print('='*60)
print('INTERPRETATION')
print('='*60)
print(f'\nprescribed:         {means["prescribed"]:.6f}')
print(f'rotated_prescribed: {means["rotated_prescribed"]:.6f}')
print(f'random_fixed:       {means["random_fixed"]:.6f}')
print(f'free:               {means["free"]:.6f}')

# Key test 1: stability
if means['random_fixed'] < means['free'] * 0.9:
    pct = (means['free'] - means['random_fixed']) / means['free'] * 100
    print(f'\n✅ STABILITY HELPS: random_fixed < free by {pct:.0f}%')
elif means['random_fixed'] > means['free'] * 1.1:
    print(f'\n❌ STABILITY ALONE HURTS: random_fixed > free')
else:
    print(f'\n⚠️  STABILITY NEUTRAL: random_fixed ≈ free')

# Key test 2: rotation invariance
r = means['rotated_prescribed'] / means['prescribed']
if r < 2.0:
    print(f'✅ ROTATION INVARIANT: rotated ≈ prescribed ({r:.2f}x)')
else:
    print(f'⚠️  ROTATION MATTERS: rotated >> prescribed ({r:.2f}x)')

# Decomposition
print(f'\n--- Decomposition ---')
total = means['free'] / means['prescribed']
stab = means['free'] / means['random_fixed'] if means['random_fixed'] > 0 else float('inf')
align = means['random_fixed'] / means['prescribed'] if means['prescribed'] > 0 else float('inf')
print(f'Total gap:                    {total:.1f}x')
print(f'Stability factor (free/rand): {stab:.1f}x')
print(f'Alignment factor (rand/presc):{align:.1f}x')
print(f'Product check:                {stab*align:.1f}x (should ≈ {total:.1f}x)')

In [ ]:
# === SAVE ===

final = {
    'config': {'seeds': SEEDS, 'episodes': N_EP, 'epochs': N_EPOCHS,
               'conditions': CONDITIONS, 'sigreg_lambda': 0.09,
               'note': 'v2 — fixed normalization in random_fixed'},
    'means': means,
    'ratios': {
        'free_vs_prescribed': means['free']/means['prescribed'],
        'random_fixed_vs_prescribed': means['random_fixed']/means['prescribed'],
        'rotated_vs_prescribed': means['rotated_prescribed']/means['prescribed'],
        'free_vs_random_fixed': means['free']/means['random_fixed'],
    },
    'per_seed': {k: {kk:vv for kk,vv in v.items() if kk!='history'} for k,v in R.items()},
    'full': R,
}
with open(f'{SAVE_DIR}/random_fixed_v2_results.json', 'w') as f:
    json.dump(final, f, indent=2)
print(f'Saved to {SAVE_DIR}/random_fixed_v2_results.json')

In [ ]:
# === FIGURE ===

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

colors = {'prescribed':'#2166AC', 'rotated_prescribed':'#67A9CF',
          'random_fixed':'#F4A582', 'free':'#D6604D'}
labels = {'prescribed':'Prescribed\n(stable+aligned)',
          'rotated_prescribed':'Rotated prescribed\n(stable+aligned, rotated)',
          'random_fixed':'Random fixed\n(stable, no alignment)',
          'free':'Free\n(drifting)'}

# Bar chart
ax = axes[0]
x = range(len(CONDITIONS))
vals = [means[c] for c in CONDITIONS]
stds = []
for c in CONDITIONS:
    sv = [R[f'{c}_seed{s}']['best_val_loss'] for s in SEEDS]
    stds.append(np.std(sv))
bars = ax.bar(x, vals, color=[colors[c] for c in CONDITIONS],
              yerr=stds, capsize=5)
ax.set_xticks(x)
ax.set_xticklabels([labels[c] for c in CONDITIONS], fontsize=7)
ax.set_ylabel('Best validation loss')
ax.set_title('Control Experiment: Stability vs Alignment', fontweight='bold')
ax.set_yscale('log')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.3,
            f'{v:.5f}', ha='center', fontsize=7)
ax.grid(True, alpha=0.15, axis='y')

# Training curves (all seeds, one condition per color)
ax = axes[1]
for cond in CONDITIONS:
    for i, seed in enumerate(SEEDS):
        h = R[f'{cond}_seed{seed}']['history']
        ax.plot([p['ep'] for p in h], [p['vp'] for p in h],
                color=colors[cond], linewidth=0.8, alpha=0.4)
    # Mean line
    all_h = [R[f'{cond}_seed{s}']['history'] for s in SEEDS]
    mean_vp = [np.mean([all_h[j][ep]['vp'] for j in range(3)]) for ep in range(N_EPOCHS)]
    ax.plot(range(1, N_EPOCHS+1), mean_vp, color=colors[cond],
            linewidth=2, label=cond)
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation loss')
ax.set_title('Training curves (3 seeds each)', fontweight='bold')
ax.set_yscale('log'); ax.legend(fontsize=7); ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/random_fixed_v2_figure.png', dpi=200)
plt.show()
print('Figure saved')